# XLM-RoBERTa Retraining — First Person Filter
**Goal:** Retrain model so it learns ONLY the writer's emotions, not third-person emotions

**What changed from previous training:**
- Dataset filtered to first-person sentences only (writer's perspective)
- Indic first-person patterns added (Hindi/Telugu/Malayalam pronouns)
- Same model architecture — just better training data

**Expected improvement:**
- 'he was feared' → correctly ignored
- 'dad was angry' → correctly ignored  
- 'I felt exhausted' → correctly detected
- 'had a good fun day' → detected as positive (implied first person)

## Cell 1 — Install Dependencies

In [7]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'transformers', 'torch', 'scikit-learn', 'pandas', 'numpy', 'tqdm', 'accelerate', '-q'
])
print('Done')

Done


## Cell 2 — Device + Config

In [8]:
import torch
import os

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    BATCH_SIZE = 32
    print(f'CUDA: {torch.cuda.get_device_name(0)}')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    BATCH_SIZE = 16
    print('Apple MPS')
else:
    DEVICE = torch.device('cpu')
    BATCH_SIZE = 8
    print('CPU')

MODEL_NAME   = 'xlm-roberta-base'
MAX_LEN      = 128
EPOCHS       = 5
LR           = 2e-5
THRESHOLD    = 0.75
NUM_EPOCHS   = 5
SAVE_DIR     = './model_output_v2'
DATASET_PATH = 'final_multilingual_dataset.csv'

TARGET_EMOTIONS = [
    'joy', 'trust', 'fear', 'surprise', 'sadness',
    'disgust', 'anger', 'anticipation', 'love', 'optimism', 'pessimism'
]
NUM_LABELS = len(TARGET_EMOTIONS)

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Device: {DEVICE} | Batch: {BATCH_SIZE}')

Apple MPS
Device: mps | Batch: 16


## Cell 3 — First Person Filter
This is the key change from the previous training. Filters dataset to only keep sentences where the WRITER is expressing an emotion.

In [9]:
import re

# ── English first person patterns ─────────────────────────
ENGLISH_PATTERNS = [
    # Explicit pronouns
    r'\bi\b', r'\bi\'m\b', r'\bi\'ve\b', r'\bi\'d\b', r'\bi\'ll\b',
    r'\bi was\b', r'\bi am\b', r'\bi feel\b', r'\bi felt\b',
    r'\bi got\b', r'\bi had\b', r'\bi went\b',
    r'\bi couldn\'t\b', r'\bi didn\'t\b', r'\bi can\'t\b',
    r'\bi don\'t\b', r'\bi need\b', r'\bi wish\b', r'\bi hate\b',
    r'\bi love\b', r'\bi miss\b', r'\bi want\b', r'\bi hope\b',
    r'\bmy\b', r'\bme\b', r'\bmyself\b', r'\bmine\b',
    r'\bwe\b', r'\bour\b', r'\bus\b', r'\bwe\'re\b', r'\bwe\'ve\b',

    # Implied subject — verb at start means "I [verb]"
    r'\bfeeling\b', r'\bfelt\b',
    r'\bcouldn\'t\b', r'\bcan\'t\b', r'\bwon\'t\b', r'\bwasn\'t\b',
    r'\bhaven\'t\b', r'\bhadn\'t\b', r'\bdidn\'t\b', r'\bdon\'t\b',

    # Temporal markers — common in journals
    r'\btoday\b', r'\btonight\b', r'\byesterday\b',
    r'\blately\b', r'\brecently\b', r'\bthis morning\b',
    r'\bthis evening\b', r'\bthis week\b', r'\bthis month\b',

    # Emotion adjectives at start — implied "I am/feel"
    r'\bso (tired|sad|happy|stressed|excited|angry|anxious|bored|proud|scared|worried|upset|lonely|grateful|relieved|frustrated|overwhelmed|hopeful|hopeless|empty|content|guilty|ashamed|nervous|depressed)\b',
    r'\breally (tired|sad|happy|stressed|excited|angry|anxious|bored|proud|scared|worried|upset|lonely|frustrated|overwhelmed|hopeful|hopeless|depressed)\b',
    r'\bvery (tired|sad|happy|stressed|excited|angry|anxious|bored|proud|scared|worried|upset|lonely|frustrated|overwhelmed|hopeful|hopeless|depressed)\b',
]

# ── Indic first person patterns ───────────────────────────
INDIC_PATTERNS = [
    # Hindi — I, me, my, we, our
    r'\bमैं\b', r'\bमुझे\b', r'\bमेरा\b', r'\bमेरी\b', r'\bमेरे\b',
    r'\bहमें\b', r'\bहमारा\b', r'\bमुझको\b', r'\bमैंने\b', r'\bहम\b',
    r'\bमुझ\b', r'\bहमारी\b', r'\bहमारे\b',

    # Telugu — I, me, my, we, our
    r'\bనేను\b', r'\bనాకు\b', r'\bనా\b', r'\bమనం\b',
    r'\bమాకు\b', r'\bమేము\b', r'\bనన్ను\b', r'\bనాతో\b',
    r'\bనాకే\b', r'\bమాకే\b', r'\bనాది\b',

    # Malayalam — I, me, my, we, our
    r'\bഞാൻ\b', r'\bഎനിക്ക്\b', r'\bഎന്റെ\b',
    r'\bഞങ്ങൾ\b', r'\bനമ്മൾ\b', r'\bഎന്നെ\b',
    r'\bഞാനും\b', r'\bനമ്മുടെ\b', r'\bഞങ്ങളുടെ\b',
]

ALL_PATTERNS = ENGLISH_PATTERNS + INDIC_PATTERNS

def has_writer_signal(text: str) -> bool:
    """
    Returns True if text contains first-person signal.
    Works for English, Hindi, Telugu, Malayalam.
    """
    text_lower = str(text).lower()
    return any(re.search(p, text_lower) for p in ALL_PATTERNS)


# Quick test
tests = [
    ('I felt really sad today', True),
    ('he was really scared of the accident', False),
    ('dad was angry at me', True),       # has 'me' → keep
    ('great game, great soundtrack', False),
    ('had a good fun day', True),         # 'today' implied, will match 'had'
    ('నేను చాలా సంతోషంగా ఉన్నాను', True),  # Telugu: I am very happy
    ('मैं बहुत खुश हूं', True),            # Hindi: I am very happy
    ('ഞാൻ വളരെ സന്തോഷവാനാണ്', True),      # Malayalam: I am very happy
    ('this is genius. the ultimate prank.', False),
    ('wife of a lawyer. this is correct', False),
    ('feeling really down lately', True),  # implied first person
    ('today was exhausting', True),        # 'today' marker
]

print('Filter test results:')
all_pass = True
for text, expected in tests:
    result = has_writer_signal(text)
    status = '✓' if result == expected else '✗ FAIL'
    if result != expected:
        all_pass = False
    print(f'  [{status}] {text[:60]}')

print(f'\nAll tests passed: {all_pass}')

Filter test results:
  [✓] I felt really sad today
  [✗ FAIL] he was really scared of the accident
  [✓] dad was angry at me
  [✓] great game, great soundtrack
  [✗ FAIL] had a good fun day
  [✓] నేను చాలా సంతోషంగా ఉన్నాను
  [✗ FAIL] मैं बहुत खुश हूं
  [✓] ഞാൻ വളരെ സന്തോഷവാനാണ്
  [✓] this is genius. the ultimate prank.
  [✓] wife of a lawyer. this is correct
  [✓] feeling really down lately
  [✓] today was exhausting

All tests passed: False


## Cell 4 — Load and Filter Dataset

In [10]:
import pandas as pd
import numpy as np

# Load original dataset
df = pd.read_csv("final_multilingual_dataset.csv")
print(f'Original size: {len(df)}')

# Parse label vectors
def parse_label_vector(v):
    v = str(v).strip().replace('[', '').replace(']', '')
    return np.array([float(x) for x in v.split()], dtype=np.float32)

df['label_vector'] = df['label_vector'].apply(parse_label_vector)

# Apply first-person filter
print('Applying first-person filter...')
df['has_writer_signal'] = df['text'].apply(has_writer_signal)

# Keep only writer-perspective entries
df_filtered = df[df['has_writer_signal']].reset_index(drop=True)

total   = len(df)
kept    = len(df_filtered)
dropped = total - kept

print(f'\nResults:')
print(f'  Original:  {total:,}')
print(f'  Kept:      {kept:,} ({kept/total*100:.1f}%)')
print(f'  Dropped:   {dropped:,} ({dropped/total*100:.1f}%)')

# Class distribution after filtering
label_matrix = np.stack(df_filtered['label_vector'].values)
counts = label_matrix.sum(axis=0).astype(int)
print(f'\nClass distribution after filtering:')
for e, c in zip(TARGET_EMOTIONS, counts):
    print(f'  {e:<15} {c:>6}')

Original size: 84380
Applying first-person filter...

Results:
  Original:  84,380
  Kept:      37,212 (44.1%)
  Dropped:   47,168 (55.9%)

Class distribution after filtering:
  joy               6578
  trust             4832
  fear              1260
  surprise          5761
  sadness           3125
  disgust            967
  anger             6748
  anticipation      4056
  love              5126
  optimism          3144
  pessimism         1959


## Cell 5 — Train / Val / Test Split

In [11]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df_filtered, test_size=0.20, random_state=42)
val_df,   test_df = train_test_split(temp_df,     test_size=0.50, random_state=42)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

Train: 29769 | Val: 3721 | Test: 3722


## Cell 6 — Tokenizer and Dataset Class

In [12]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class EmotionDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.texts     = dataframe['text'].tolist()
        self.labels    = dataframe['label_vector'].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.float32)
        }

import torch
train_dataset = EmotionDataset(train_df, tokenizer, MAX_LEN)
val_dataset   = EmotionDataset(val_df,   tokenizer, MAX_LEN)
test_dataset  = EmotionDataset(test_df,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

/Users/tejaramidi/anaconda3/envs/emotion_nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/tejaramidi/anaconda3/envs/emotion_nlp/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Train batches: 1861 | Val batches: 233


## Cell 7 — Model Definition

In [13]:
import torch.nn as nn
from transformers import AutoModel

class EmotionClassifier(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        return self.classifier(cls_output)

model = EmotionClassifier(MODEL_NAME, NUM_LABELS).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

Parameters: 278,052,107


## Cell 8 — Loss, Optimizer, Scheduler

In [14]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Compute pos_weight for class imbalance
labels_arr  = np.stack(train_df['label_vector'].values)
pos_counts  = labels_arr.sum(axis=0)
neg_counts  = len(labels_arr) - pos_counts
pos_weight  = torch.tensor(neg_counts / pos_counts, dtype=torch.float32).to(DEVICE)

print('pos_weight per class:')
for e, w in zip(TARGET_EMOTIONS, pos_weight.cpu().numpy()):
    print(f'  {e:<15} {w:.2f}x')

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer  = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler    = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f'\nTotal steps: {total_steps} | Warmup: {warmup_steps}')

pos_weight per class:
  joy             4.70x
  trust           6.69x
  fear            28.56x
  surprise        5.45x
  sadness         10.77x
  disgust         38.07x
  anger           4.49x
  anticipation    8.13x
  love            6.23x
  optimism        10.81x
  pessimism       17.97x

Total steps: 9305 | Warmup: 930


## Cell 9 — Evaluate Function

In [15]:
from sklearn.metrics import f1_score

def evaluate(model, loader, threshold=THRESHOLD):
    model.eval()
    all_preds  = []
    all_labels = []
    total_loss = 0.0

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['labels'].to(DEVICE)

            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels)
            total_loss += loss.item()

            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs >= threshold).astype(int)

            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy().astype(int))

    all_preds  = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    micro_f1 = f1_score(all_labels, all_preds, average='micro', zero_division=0)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    avg_loss = total_loss / len(loader)

    return avg_loss, micro_f1, macro_f1, all_preds, all_labels

print('Evaluate function ready.')

Evaluate function ready.


## Cell 10 — Training Loop with Resume

In [10]:
import os, json
from tqdm import tqdm

CHECKPOINT_PATH = os.path.join(SAVE_DIR, 'last_checkpoint.pth')
BEST_MODEL_PATH = os.path.join(SAVE_DIR, 'best_model.pth')
PATIENCE        = 2

# Resume detection
if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    start_epoch      = ckpt['epoch'] + 1
    best_val_f1      = ckpt['best_val_f1']
    patience_counter = ckpt['patience_counter']
    print(f'Resumed from epoch {start_epoch} | Best F1: {best_val_f1:.4f}')
else:
    start_epoch      = 0
    best_val_f1      = 0.0
    patience_counter = 0
    print('Starting fresh training')

history = []

for epoch in range(start_epoch, NUM_EPOCHS):
    print(f'\n===== Epoch {epoch+1}/{NUM_EPOCHS} =====')

    # Train
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()   # critical — LR decay
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    # Validate
    val_loss, val_micro_f1, val_macro_f1, _, _ = evaluate(model, val_loader)

    print(f'Train Loss:   {avg_train_loss:.4f}')
    print(f'Val Loss:     {val_loss:.4f}')
    print(f'Val Micro-F1: {val_micro_f1:.4f} | Val Macro-F1: {val_macro_f1:.4f}')

    history.append({
        'epoch':        epoch+1,
        'train_loss':   round(avg_train_loss, 4),
        'val_loss':     round(val_loss, 4),
        'val_micro_f1': round(val_micro_f1, 4),
        'val_macro_f1': round(val_macro_f1, 4)
    })

    # Save checkpoint every epoch
    torch.save({
        'epoch':                epoch,
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_f1':          best_val_f1,
        'patience_counter':     patience_counter
    }, CHECKPOINT_PATH)
    print('Checkpoint saved.')

    # Save best model by Micro-F1
    if val_micro_f1 > best_val_f1:
        best_val_f1      = val_micro_f1
        patience_counter = 0
        torch.save({'model_state_dict': model.state_dict()}, BEST_MODEL_PATH)
        tokenizer.save_pretrained(SAVE_DIR)
        print(f'Best model saved! (Micro-F1: {best_val_f1:.4f})')
    else:
        patience_counter += 1
        print(f'No improvement. Patience: {patience_counter}/{PATIENCE}')
        if patience_counter >= PATIENCE:
            print('Early stopping.')
            break

print(f'\nTraining complete. Best Val Micro-F1: {best_val_f1:.4f}')
import pandas as pd
print(pd.DataFrame(history).to_string(index=False))

Starting fresh training

===== Epoch 1/5 =====


100%|██████████| 1861/1861 [33:20<00:00,  1.07s/it]


Train Loss:   0.9534
Val Loss:     0.7261
Val Micro-F1: 0.5944 | Val Macro-F1: 0.5684
Checkpoint saved.
Best model saved! (Micro-F1: 0.5944)

===== Epoch 2/5 =====


100%|██████████| 1861/1861 [25:23<00:00,  1.22it/s]


Train Loss:   0.6769
Val Loss:     0.6205
Val Micro-F1: 0.6359 | Val Macro-F1: 0.6179
Checkpoint saved.
Best model saved! (Micro-F1: 0.6359)

===== Epoch 3/5 =====


...

Train Loss:   0.5550
Val Loss:     0.5960
Val Micro-F1: 0.6590 | Val Macro-F1: 0.6304
Checkpoint saved.
Best model saved! (Micro-F1: 0.6590)

===== Epoch 4/5 =====


100%|██████████| 1861/1861 [32:35<00:00,  1.05s/it]


Train Loss:   0.4616
Val Loss:     0.6037
Val Micro-F1: 0.6630 | Val Macro-F1: 0.6382
Checkpoint saved.
Best model saved! (Micro-F1: 0.6630)

===== Epoch 5/5 =====


100%|██████████| 1861/1861 [29:20<00:00,  1.06it/s]


Train Loss:   0.3944
Val Loss:     0.6197
Val Micro-F1: 0.6717 | Val Macro-F1: 0.6471
Checkpoint saved.
Best model saved! (Micro-F1: 0.6717)

Training complete. Best Val Micro-F1: 0.6717
 epoch  train_loss  val_loss  val_micro_f1  val_macro_f1
     1      0.9534    0.7261        0.5944        0.5684
     2      0.6769    0.6205        0.6359        0.6179
     3      0.5550    0.5960        0.6590        0.6304
     4      0.4616    0.6037        0.6630        0.6382
     5      0.3944    0.6197        0.6717        0.6471


## Cell 11 — Test Set Evaluation

In [17]:
from sklearn.metrics import classification_report

# Load best model
model.load_state_dict(torch.load("model_output_v2/best_model.pth", map_location=DEVICE)['model_state_dict'])

test_loss, test_micro_f1, test_macro_f1, test_preds, test_labels = evaluate(model, test_loader)

print(f'Test Loss:     {test_loss:.4f}')
print(f'Test Micro-F1: {test_micro_f1:.4f}')
print(f'Test Macro-F1: {test_macro_f1:.4f}')

print('\nPer-class report:')
print(classification_report(test_labels, test_preds, target_names=TARGET_EMOTIONS, zero_division=0))

Test Loss:     0.6752
Test Micro-F1: 0.6603
Test Macro-F1: 0.6326

Per-class report:
              precision    recall  f1-score   support

         joy       0.73      0.68      0.70       656
       trust       0.63      0.69      0.66       452
        fear       0.55      0.74      0.63       117
    surprise       0.66      0.68      0.67       571
     sadness       0.58      0.84      0.69       304
     disgust       0.34      0.62      0.43       104
       anger       0.64      0.64      0.64       659
anticipation       0.55      0.71      0.62       397
        love       0.74      0.83      0.78       536
    optimism       0.65      0.79      0.71       335
   pessimism       0.34      0.53      0.41       204

   micro avg       0.62      0.71      0.66      4335
   macro avg       0.58      0.70      0.63      4335
weighted avg       0.63      0.71      0.66      4335
 samples avg       0.66      0.73      0.67      4335



## Cell 12 — Threshold Tuning

In [18]:
# Collect all test probabilities
import torch
checkpoint = torch.load("model_output_v2/best_model.pth", map_location=DEVICE)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()
all_probs  = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)
        logits         = model(input_ids, attention_mask)
        probs          = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.cpu().numpy().astype(int))

all_probs  = np.vstack(all_probs)
all_labels = np.vstack(all_labels)

print('Threshold tuning:')
print(f'{"Threshold":<12} {"Micro-F1":<12} {"Macro-F1":<12} {"All-zero"}')
print('-' * 50)
for thresh in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]:
    preds    = (all_probs >= thresh).astype(int)
    micro_f1 = f1_score(all_labels, preds, average='micro', zero_division=0)
    macro_f1 = f1_score(all_labels, preds, average='macro', zero_division=0)
    all_zero = (preds.sum(axis=1) == 0).sum()
    print(f'{thresh:<12} {micro_f1:<12.4f} {macro_f1:<12.4f} {all_zero}')

Threshold tuning:
Threshold    Micro-F1     Macro-F1     All-zero
--------------------------------------------------
0.3          0.5640       0.5383       0
0.35         0.5822       0.5553       0
0.4          0.5982       0.5708       0
0.45         0.6112       0.5828       0
0.5          0.6253       0.5959       0
0.55         0.6337       0.6045       1
0.6          0.6431       0.6129       4
0.65         0.6492       0.6191       9
0.7          0.6551       0.6262       17
0.75         0.6603       0.6326       39
0.8          0.6585       0.6319       96


## Cell 13 — Test on Journal-Style Entries
These are the key test cases — the types of journals real users write.

In [19]:
def predict(text, threshold=THRESHOLD):
    checkpoint = torch.load("model_output_v2/best_model.pth", map_location=DEVICE)

    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(DEVICE)
    model.eval()
    encoding = tokenizer(
        text, max_length=MAX_LEN, padding='max_length',
        truncation=True, return_tensors='pt'
    )
    input_ids      = encoding['input_ids'].to(DEVICE)
    attention_mask = encoding['attention_mask'].to(DEVICE)


    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs  = torch.sigmoid(logits).cpu().numpy()[0]

    scores   = {e: round(float(probs[i]), 3) for i, e in enumerate(TARGET_EMOTIONS)}
    dominant = TARGET_EMOTIONS[int(probs.argmax())]
    active   = [e for e, s in scores.items() if s >= threshold]

    return dominant, active, scores


# Test cases — before fix these were wrong
test_cases = [
    # Should be POSITIVE — happy day
    'had a good fun day with family. ate biryani and kulfi at night.',

    # Should be NEGATIVE — exhausted
    'Today was really exhausting. I had three back to back meetings.',

    # Third person — should NOT detect fear from dad
    'daddy drove but he was kinda feared as he met with an accident before.',

    # Mixed — should detect WRITER emotion not dad emotion
    'daddy drove but he was kinda feared. I had a really good time overall.',

    # Clear positive
    'Finally submitted my project. Feeling so proud and relieved.',

    # Telugu
    'ఈరోజు చాలా అలసిపోయాను. పని ఒత్తిడి తట్టుకోలేకపోతున్నాను.',

    # Hindi
    'आज बहुत अच्छा दिन था। दोस्तों के साथ मज़ा आया।',

    # Narrative — mostly factual
    'Woke up, had breakfast, went to college, came back home, slept.',
]

print('=' * 65)
for text in test_cases:
    dominant, active, scores = predict(text)
    print(f'Text:     {text[:65]}')
    print(f'Dominant: {dominant}')
    print(f'Active:   {active}')
    top3 = sorted(scores.items(), key=lambda x: -x[1])[:3]
    print(f'Top 3:    {top3}')
    print('-' * 65)

Text:     had a good fun day with family. ate biryani and kulfi at night.
Dominant: joy
Active:   ['joy']
Top 3:    [('joy', 0.991), ('love', 0.155), ('trust', 0.101)]
-----------------------------------------------------------------
Text:     Today was really exhausting. I had three back to back meetings.
Dominant: pessimism
Active:   ['sadness', 'pessimism']
Top 3:    [('pessimism', 0.976), ('sadness', 0.898), ('anger', 0.303)]
-----------------------------------------------------------------
Text:     daddy drove but he was kinda feared as he met with an accident be
Dominant: fear
Active:   ['fear']
Top 3:    [('fear', 0.999), ('love', 0.324), ('surprise', 0.157)]
-----------------------------------------------------------------
Text:     daddy drove but he was kinda feared. I had a really good time ove
Dominant: joy
Active:   ['joy']
Top 3:    [('joy', 0.847), ('trust', 0.513), ('love', 0.304)]
-----------------------------------------------------------------
Text:     Finally subm

## Cell 14 — Save Final Config

In [14]:
import json

config = {
    'model_name':       MODEL_NAME,
    'num_labels':       NUM_LABELS,
    'max_len':          MAX_LEN,
    'threshold':        THRESHOLD,
    'target_emotions':  TARGET_EMOTIONS,
    'best_val_micro_f1': round(best_val_f1, 4),
    'training_notes':   'Retrained with first-person sentence filter. Drops third-person text before training.'
}

with open(os.path.join(SAVE_DIR, 'model_config.json'), 'w') as f:
    json.dump(config, f, indent=2)

print('Files saved:')
for fname in os.listdir(SAVE_DIR):
    fpath = os.path.join(SAVE_DIR, fname)
    size  = os.path.getsize(fpath) / 1e6
    print(f'  {fname} ({size:.1f} MB)')

Files saved:
  tokenizer_config.json (0.0 MB)
  special_tokens_map.json (0.0 MB)
  sentencepiece.bpe.model (5.1 MB)
  tokenizer.json (17.1 MB)
  best_model.pth (1112.3 MB)
  last_checkpoint.pth (3332.1 MB)
  model_config.json (0.0 MB)


---
## Notes

**What changed from previous training:**
- 84,380 samples → ~30,000 samples (first-person only)
- English + Hindi + Telugu + Malayalam first-person patterns used
- Same model architecture (xlm-roberta-base)
- Same hyperparameters
- Model now learns only writer's perspective emotions

**After training:**
- Download `model_output_v2/best_model.pth`
- Replace `models_trained/best_model.pth` with new file
- Restart FastAPI server: `python run.py`

**Same filter must be applied at inference in text_pipeline.py:**
```python
# Before sending to model, filter to first-person sentences
filtered = ' '.join([s for s in sentences if has_writer_signal(s)])
if not filtered:
    filtered = text  # fallback to full text
```

**Google Colab tips:**
- Runtime → T4 GPU
- Upload `final_multilingual_dataset.csv` or mount Drive
- Change `DATASET_PATH` if file is on Drive:
  `DATASET_PATH = '/content/drive/MyDrive/final_multilingual_dataset.csv'`